# Apartado 1 — Implementación de Modelos y Evaluación con Hold-Out

En este apartado implementamos **tres modelos personalizados** tal como indica el enunciado:

- `myLinearRegression`
- `myLogisticRegression`
- `myDecisionTreeClassifier` (ID3 + Gini)

El objetivo es:
1. Implementar cada modelo desde cero, siguiendo la especificación dada en la teoría.
2. Hacerlos compatibles con scikit-learn, heredando de las clases adecuadas.
3. Evaluarlos mediante **10 particiones hold-out** (75% entrenamiento, 25% test).
4. Calcular métricas de rendimiento y obtener resultados promedio y desviación estándar.
5. En el caso del árbol, incluir también una **visualización textual** del árbol y la **extracción de reglas**.

A continuación cargamos y preparamos los datasets necesarios.


In [10]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, RegressorMixin, ClassifierMixin
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.preprocessing import LabelEncoder


### Carga del dataset de Regresión

Cargamos el dataset `datos_regresion_20_muestras.csv`.  
Contiene 20 muestras y 3 atributos numéricos.

Se separan:
- `X_reg` → atributos
- `y_reg` → variable objetivo

Este dataset se usará para evaluar `myLinearRegression`.


In [13]:
df_reg = pd.read_csv("datos_regresion_20_muestras.csv")

print("Dataset de REGRESIÓN:")
display(df_reg)

X_reg = df_reg.drop(columns=["Precio_Venta"]).values
y_reg = df_reg["Precio_Venta"].values


Dataset de REGRESIÓN:


,Tamaño_m2,Num_Pisos,Edad_Años,Precio_Venta
0,100,2,10,256.4
1,150,3,5,401.1
2,80,1,15,185.7
3,120,2,8,305.9
4,90,3,12,238.2
5,180,4,3,489.6
6,70,1,20,150.1
7,110,2,6,278.3
8,130,3,4,360.5
9,60,1,25,120.9


### Carga del dataset de Clasificación

Cargamos el dataset `datos_clasificacion_20_muestras.csv`.  
Contiene 20 muestras, 3 atributos numéricos y una etiqueta binaria.

Se utilizará para evaluar `myLogisticRegression`.


In [15]:
df_clf = pd.read_csv("datos_clasificacion_20_muestras.csv")

print("Dataset de CLASIFICACIÓN:")
display(df_clf)

X_clf = df_clf.drop(columns=["Aprobado"]).values
y_clf = df_clf["Aprobado"].values


Dataset de CLASIFICACIÓN:


,Puntaje_Examen,Horas_Estudio,Asistencias,Aprobado
0,75,10,95,1
1,55,4,70,0
2,85,15,100,1
3,40,2,60,0
4,65,8,85,1
5,70,7,90,1
6,35,3,55,0
7,90,18,98,1
8,60,6,80,1
9,50,5,65,0


### Carga del dataset categórico para ID3 (tenis_tree.csv)

ID3 solo funciona correctamente con atributos categóricos.  
Por ello utilizamos el dataset clásico Play Tennis.

Cada columna se codifica con un LabelEncoder independiente.


In [33]:
df_tree = pd.read_csv("tenis_tree.csv")

# 🔥 Eliminar la columna Día (ID que rompe ID3)
df_tree = df_tree.drop(columns=["Día"])

print("Dataset tenis_tree.csv corregido:")
display(df_tree)

# Separar atributos y etiqueta
X_cat = df_tree.iloc[:, :-1]
y_cat = df_tree.iloc[:, -1]

# Codificar atributos categóricos
label_encoders = {}
X_tree = X_cat.copy()

for col in X_cat.columns:
    le = LabelEncoder()
    X_tree[col] = le.fit_transform(X_tree[col])
    label_encoders[col] = le

# Codificar la etiqueta
y_le = LabelEncoder()
y_tree = y_le.fit_transform(y_cat)

feature_names_tree = list(df_tree.columns[:-1])
X_tree = X_tree.values



Dataset tenis_tree.csv corregido:


,Pronóstico,Temperatura,Humedad,Viento,Jugar Tenis
0,Soleado,Caluroso,Alta,Débil,No
1,Soleado,Caluroso,Alta,Fuerte,No
2,Nublado,Caluroso,Alta,Débil,Sí
3,Lluvia,Templado,Alta,Débil,Sí
4,Lluvia,Frío,Normal,Débil,Sí
5,Lluvia,Frío,Normal,Fuerte,No
6,Nublado,Frío,Normal,Fuerte,Sí
7,Soleado,Templado,Alta,Débil,No
8,Soleado,Frío,Normal,Débil,Sí
9,Lluvia,Templado,Normal,Débil,Sí


## Implementación de myLinearRegression

Implementación de regresión lineal mediante la solución cerrada.  
Hereda de `BaseEstimator` y `RegressorMixin`.

Métodos implementados:
- `fit()`
- `predict()`


In [34]:
class myLinearRegression(BaseEstimator, RegressorMixin):
    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept
        self.coef_ = None
        self.intercept_ = None

    def _add_intercept(self, X):
        if not self.fit_intercept:
            return X
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, float).reshape(-1, 1)

        Xd = self._add_intercept(X)
        w = np.linalg.pinv(Xd.T @ Xd) @ Xd.T @ y

        if self.fit_intercept:
            self.intercept_ = w[0, 0]
            self.coef_ = w[1:, 0]
        else:
            self.intercept_ = 0
            self.coef_ = w[:, 0]

        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_


## Implementación de myLogisticRegression

Regresión logística binaria implementada desde cero usando descenso de gradiente.  
Incluye sigmoide estable.

Hereda de:
- `BaseEstimator`
- `ClassifierMixin`


In [42]:
class myLogisticRegression(BaseEstimator, ClassifierMixin):
    def __init__(self, lr=0.05, max_iter=2000, tol=1e-4):
        self.lr = lr
        self.max_iter = max_iter
        self.tol = tol
        self.w_ = None

    def _add_intercept(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def _sigmoid(self, z):
       
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, float)

        Xd = self._add_intercept(X)
        n_samples, n_features = Xd.shape

        rng = np.random.default_rng()
        self.w_ = rng.uniform(-0.01, 0.01, size=n_features)

        for _ in range(self.max_iter):
            z = Xd @ self.w_
            p = self._sigmoid(z)

            grad = -(y - p) @ Xd / n_samples
            new_w = self.w_ - self.lr * grad

            if np.linalg.norm(new_w - self.w_) < self.tol:
                self.w_ = new_w
                break

            self.w_ = new_w

        return self

    def predict_proba(self, X):
        Xd = self._add_intercept(np.asarray(X, float))
        p = self._sigmoid(Xd @ self.w_)
        return np.vstack([1 - p, p]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


## Implementación de myDecisionTreeClassifier (ID3 + Gini)

Árbol de decisión basado en:
- Entropía (ID3)
- Gini (alternativa)

Los atributos son categóricos y se dividen por igualdad.


In [43]:
class myDecisionTreeClassifier(BaseEstimator, ClassifierMixin):

    class Node:
        def __init__(self, feature=None, children=None, prediction=None):
            self.feature = feature
            self.children = children or {}
            self.prediction = prediction

    def __init__(self, criterion="entropy"):
        assert criterion in ("entropy", "gini")
        self.criterion = criterion
        self.root_ = None

    def _entropy(self, y):
        _, counts = np.unique(y, return_counts=True)
        p = counts / counts.sum()
        return -np.sum(p * np.log2(p + 1e-9))

    def _gini(self, y):
        _, counts = np.unique(y, return_counts=True)
        p = counts / counts.sum()
        return 1 - np.sum(p ** 2)

    def _impurity(self, y):
        return self._entropy(y) if self.criterion == "entropy" else self._gini(y)

    def _best_split(self, X, y, features):
        base_imp = self._impurity(y)
        best_gain = -1
        best_feature = None
        n = len(y)

        for f in features:
            vals = np.unique(X[:, f])
            imp_after = 0
            for v in vals:
                mask = (X[:, f] == v)
                y_sub = y[mask]
                imp_after += (len(y_sub) / n) * self._impurity(y_sub)

            gain = base_imp - imp_after
            if gain > best_gain:
                best_gain = gain
                best_feature = f

        return best_feature, best_gain

    def _build(self, X, y, features):
        classes, counts = np.unique(y, return_counts=True)
        majority = classes[np.argmax(counts)]

        if len(classes) == 1:
            return self.Node(prediction=classes[0])

        if not features:
            return self.Node(prediction=majority)

        best_feature, gain = self._best_split(X, y, features)

        if best_feature is None or gain <= 0:
            return self.Node(prediction=majority)

        node = self.Node(feature=best_feature)
        vals = np.unique(X[:, best_feature])
        remaining = [f for f in features if f != best_feature]

        for v in vals:
            mask = X[:, best_feature] == v
            node.children[v] = self._build(X[mask], y[mask], remaining)

        return node

    def fit(self, X, y):
        features = list(range(X.shape[1]))
        self.root_ = self._build(np.asarray(X), np.asarray(y), features)
        return self

    def _predict_one(self, x, node):
        if node.prediction is not None:
            return node.prediction

        v = x[node.feature]
        child = node.children.get(v)

        if child is None:
            preds = []
            for c in node.children.values():
                if c.prediction is not None:
                    preds.append(c.prediction)
            if preds:
                vals, cnt = np.unique(preds, return_counts=True)
                return vals[np.argmax(cnt)]
            return 0

        return self._predict_one(x, child)

    def predict(self, X):
        return np.array([self._predict_one(row, self.root_) for row in X])


## Visualización textual del árbol


In [44]:
def print_tree(node, feature_names, indent=""):
    if node.prediction is not None:
        print(indent + f"🔸 Predicción: {node.prediction}")
        return
    
    feat = feature_names[node.feature]
    print(indent + f"📌 Test: {feat}")

    for v, child in node.children.items():
        print(indent + f" ├── Si {feat} == {v}:")
        print_tree(child, feature_names, indent + " │   ")


## Extracción de reglas del árbol


In [45]:
def extract_rules(node, feature_names, current="", rules=None):
    if rules is None:
        rules = []

    if node.prediction is not None:
        rules.append(current + f" → Predicción = {node.prediction}")
        return rules

    feat = feature_names[node.feature]

    for val, child in node.children.items():
        new = current + f"{feat} == {val} AND "
        extract_rules(child, feature_names, new, rules)

    return rules


## Evaluación con 10 particiones Hold-Out

Se evalúan:

- Regresión → MSE, MAE, R2
- Clasificación → Accuracy, Precision, Recall, F1


In [46]:
def eval_reg(model_class, X, y, n=10, **kwargs):
    mses, maes, r2s = [], [], []
    for seed in range(n):
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed)
        model = model_class(**kwargs).fit(Xtr, ytr)
        yp = model.predict(Xte)

        mses.append(mean_squared_error(yte, yp))
        maes.append(mean_absolute_error(yte, yp))
        r2s.append(r2_score(yte, yp))

    return pd.DataFrame({
        "Métrica": ["MSE", "MAE", "R2"],
        "Media":   [np.mean(mses), np.mean(maes), np.mean(r2s)],
        "Std":     [np.std(mses), np.std(maes), np.std(r2s)]
    })


def eval_clf(model_class, X, y, n=10, **kwargs):
    accs, precs, recs, f1s = [], [], [], []
    for seed in range(n):
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed)
        model = model_class(**kwargs).fit(Xtr, ytr)
        yp = model.predict(Xte)

        accs.append(accuracy_score(yte, yp))
        precs.append(precision_score(yte, yp, zero_division=0))
        recs.append(recall_score(yte, yp, zero_division=0))
        f1s.append(f1_score(yte, yp, zero_division=0))

    return pd.DataFrame({
        "Métrica": ["Accuracy", "Precision", "Recall", "F1"],
        "Media":   [np.mean(accs), np.mean(precs), np.mean(recs), np.mean(f1s)],
        "Std":     [np.std(accs), np.std(precs), np.std(recs), np.std(f1s)]
    })


## Resultados finales de todos los modelos


In [47]:
print("=== myLinearRegression ===")
display(eval_reg(myLinearRegression, X_reg, y_reg))

print("\n=== myLogisticRegression ===")
display(eval_clf(myLogisticRegression, X_clf, y_clf, lr=0.05, max_iter=2000))

print("\n=== myDecisionTreeClassifier (ID3 – Entropía) ===")
display(eval_clf(myDecisionTreeClassifier, X_tree, y_tree, criterion="entropy"))

print("\n=== myDecisionTreeClassifier (Gini) ===")
display(eval_clf(myDecisionTreeClassifier, X_tree, y_tree, criterion="gini"))


=== myLinearRegression ===


,Métrica,Media,Std
0,MSE,478.539264,389.640362
1,MAE,13.778755,4.891418
2,R2,0.939576,0.055927



=== myLogisticRegression ===


,Métrica,Media,Std
0,Accuracy,0.840000,0.149666
1,Precision,0.926667,0.147422
2,Recall,0.833333,0.232737
3,F1,0.843095,0.160697



=== myDecisionTreeClassifier (ID3 – Entropía) ===


,Métrica,Media,Std
0,Accuracy,0.680000,0.203961
1,Precision,0.773333,0.206317
2,Recall,0.791667,0.271953
3,F1,0.741905,0.206520



=== myDecisionTreeClassifier (Gini) ===


,Métrica,Media,Std
0,Accuracy,0.660000,0.200998
1,Precision,0.768333,0.206216
2,Recall,0.766667,0.262996
3,F1,0.728016,0.200758


## Visualización del Árbol y Reglas


In [48]:
print("ÁRBOL ID3 (Entropía):\n")
model = myDecisionTreeClassifier(criterion="entropy").fit(X_tree, y_tree)
print_tree(model.root_, feature_names_tree)

print("\nREGLAS DEL ÁRBOL:\n")
for r in extract_rules(model.root_, feature_names_tree):
    print("•", r)


ÁRBOL ID3 (Entropía):

📌 Test: Pronóstico
 ├── Si Pronóstico == 0:
 │   📌 Test: Viento
 │    ├── Si Viento == 0:
 │    │   📌 Test: Temperatura
 │    │    ├── Si Temperatura == 0:
 │    │    │   🔸 Predicción: 0
 │    │    ├── Si Temperatura == 1:
 │    │    │   🔸 Predicción: 1
 │    │    ├── Si Temperatura == 2:
 │    │    │   🔸 Predicción: 1
 │    ├── Si Viento == 1:
 │    │   🔸 Predicción: 0
 ├── Si Pronóstico == 1:
 │   🔸 Predicción: 1
 ├── Si Pronóstico == 2:
 │   📌 Test: Humedad
 │    ├── Si Humedad == 0:
 │    │   🔸 Predicción: 0
 │    ├── Si Humedad == 1:
 │    │   🔸 Predicción: 1

REGLAS DEL ÁRBOL:

• Pronóstico == 0 AND Viento == 0 AND Temperatura == 0 AND  → Predicción = 0
• Pronóstico == 0 AND Viento == 0 AND Temperatura == 1 AND  → Predicción = 1
• Pronóstico == 0 AND Viento == 0 AND Temperatura == 2 AND  → Predicción = 1
• Pronóstico == 0 AND Viento == 1 AND  → Predicción = 0
• Pronóstico == 1 AND  → Predicción = 1
• Pronóstico == 2 AND Humedad == 0 AND  → Predicción = 0
• 

# Interpretación de Resultados — Apartado 1

En este apartado se han evaluado tres modelos personalizados desarrollados desde cero:
- `myLinearRegression`
- `myLogisticRegression`
- `myDecisionTreeClassifier` (criterios Entropía y Gini)

Cada modelo se ha evaluado mediante **10 particiones Hold-Out** (75% entrenamiento, 25% test),
calculando métricas y obteniendo la media y desviación estándar.

A continuación se presentan las conclusiones de cada modelo.

---

## 1. Resultados de myLinearRegression

**R² = 0.9396 ± 0.0559**  
El modelo explica aproximadamente el **94% de la varianza** de la variable objetivo, lo que indica un ajuste muy bueno para este dataset artificial con relación lineal marcada.

Los valores de error absoluto y cuadrático son:

- **MSE = 478.53 ± 389.64**  
- **MAE = 13.78 ± 4.89**

Estos valores son grandes únicamente porque la variable objetivo se encuentra en un rango numericamente elevado.
La desviación estándar también es alta debido al **pequeño tamaño del conjunto de datos (20 muestras)**: cada test contiene solo 5 muestras, lo que genera variabilidad entre particiones.

**Conclusión:**  
El modelo de regresión lineal funciona correctamente y captura muy bien la relación entre las variables.

---

## 2. Resultados de myLogisticRegression

- **Accuracy = 0.84 ± 0.15**  
- **Precision = 0.93 ± 0.15**  
- **Recall = 0.83 ± 0.23**  
- **F1 = 0.84 ± 0.16**

El modelo muestra un rendimiento sólido, con buenos valores tanto en precisión como en recall.
De nuevo, la variabilidad entre particiones es consecuencia del reducido tamaño del dataset (20 muestras).

**Conclusión:**  
La regresión logística presenta un rendimiento alto y estable, capturando adecuadamente la frontera de decisión del dataset de clasificación.

---

## 3. Resultados de myDecisionTreeClassifier (ID3 — Entropía)

- **Accuracy = 0.68 ± 0.20**  
- **Precision = 0.77 ± 0.21**  
- **Recall = 0.79 ± 0.27**  
- **F1 = 0.74 ± 0.21**

Los resultados son coherentes con las características del dataset `tenis_tree.csv`, donde:
- Los atributos son categóricos.
- La estructura del árbol no es excesivamente profunda.
- El número total de muestras es reducido.

La variabilidad de las métricas se debe a que, en cada partición Hold-Out, pueden aparecer o desaparecer combinaciones de categorías relevantes para la clasificación.

**Conclusión:**  
El árbol basado en entropía ofrece un rendimiento correcto y consistente con lo esperado para un ID3 puro sobre un dataset pequeño.

---

## 4. Resultados de myDecisionTreeClassifier (Gini)

- **Accuracy = 0.66 ± 0.20**  
- **Precision = 0.77 ± 0.21**  
- **Recall = 0.77 ± 0.26**  
- **F1 = 0.73 ± 0.20**

Estos resultados son muy similares a los obtenidos con entropía.  
Esto es habitual, ya que en datasets pequeños y bien estructurados, **Gini y Entropía suelen elegir los mismos atributos en cada división** o árboles muy parecidos.

**Conclusión:**  
El árbol con Gini tiene un comportamiento prácticamente equivalente al de entropía, confirmando que ambas medidas generan árboles similares en datasets categóricos y de tamaño reducido.

---

# Conclusión General del Apartado 1

Los tres modelos implementados funcionan correctamente y muestran un comportamiento coherente con los datasets y con la teoría:

- La **regresión lineal** modela muy bien la relación entre variables en el dataset sintético.
- La **regresión logística** obtiene un rendimiento alto a pesar del tamaño reducido del dataset.
- El **árbol de decisión ID3**, tanto con entropía como con gini, produce resultados razonables y consistentes con el comportamiento teórico del algoritmo en datos categóricos.

El conjunto de métricas obtenidas valida que las implementaciones son correctas  
y que todos los modelos funcionan como se esperaba de acuerdo al enunciado.
